# 01 · Qué observamos realmente
Cuaderno de trabajo local. Ejecutar en el entorno del repositorio, con los CSV indicados en el README. Solo se presentan resúmenes: nunca imprimir los identificadores Fitbit ni el historial individual.
    
**Preguntas:** ¿cuántas personas y días aportan datos?, ¿la cobertura es homogénea?, ¿qué podemos inferir sin inventar una relación entre fuentes?

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'backend/privacy.py').is_file())
sys.path.insert(0, str(ROOT))
RAW = ROOT / 'data/raw'
from collections import Counter
from statistics import mean
from backend.dataio import read_activity
from backend.experiment import gym_records
rows, duplicates = read_activity([RAW/'dailyActivity_merged.csv'])
coverage = Counter(u for u, d in rows)
print(f'{len(coverage)} participantes, {len(rows)} días-persona, {duplicates} filas idénticas repetidas')
print('Periodo disponible:', min(d for u,d in rows), '—', max(d for u,d in rows))
print('Cobertura (días): mínima, media y máxima', min(coverage.values()), round(mean(coverage.values()),1), max(coverage.values()))
gym = gym_records(RAW/'gym_members_exercise_tracking.csv')
print('Perfiles Gym distintos por contenido:', len(gym))

## Clasificación y selección
- **ID Fitbit:** seudonimizar con HMAC por dominio al importar. Que los días compartan seudónimo es útil para agregarlos; no los convierte en anónimos.
- **Día e ID interno:** metadatos de acceso restringido. Su combinación puede caracterizar rutinas.
- **Minutos, pulso, sueño, peso y grasa:** cifrar si se conservan para seguimiento. La tabla de investigación guarda solo pasos y minutos; no copiar todo el CSV.
- **Género declarado en Gym:** usar únicamente para comparar el comportamiento del modelo del laboratorio; no sirve para inferir el género de participantes Fitbit.
- Un día sin lectura no equivale a cero minutos. El promedio personal usa días observados; reflejará peor a quienes registran poco.
- Los perfiles ficticios del club prueban los recorridos operativos. No vincular los dos archivos por coincidencia supuesta ni evaluar predicciones sobre parejas aleatorias.

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(6,2.8))
h=Counter(coverage.values())
ax.bar(sorted(h), [h[k] for k in sorted(h)], color='#267158')
ax.set(xlabel='Días observados por persona',ylabel='Personas',title='Cobertura del archivo Fitbit')
fig.tight_layout()
plt.show()

**Discusión ética:** la selección de personas que usan wearables no es una muestra representativa de toda la clientela. Más observaciones de una persona no deben aumentar su voto en la media del grupo. Publicar una curva individual seudonimizada no es equivalente a publicar un agregado protegido. Ver S07 (privacidad), laboratorio 10b (clasificación, seudonimización y reidentificación), L11 (DPIA) y S13 (sesgo en la recolección).